# CS577 Tiny ViT — run everything on Colab

This notebook only calls `main.py` and shows results. No model or training code here, all logic is in `src/`.

Paper reference: ViT (Dosovitskiy et al., ICLR 2021)  
Author: <name>  
Adapted from: none

**Runtime:** Colab T4 GPU (Runtime → Change runtime type → GPU)

## 1. Check environment
GPU name, python and torch versions.

In [1]:
!pip freeze | grep -Ei '^(torch|torchvision|numpy|pandas|matplotlib|pytest)=='

matplotlib==3.10.0
numpy==2.4.6
pandas==2.2.3
pytest==8.4.2
torch==2.14.1+cu130
torchvision==0.29.1+cu130


In [2]:
!nvidia-smi

import pandas as pd
import platform
import torch
import torchvision
from pathlib import Path
from IPython.display import display, IFrame

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Thu Nov 26 22:45:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   54C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo and install packages

In [3]:
!git clone https://github.com/jhwang00/cs577-tiny-vit.git
%cd cs577-tiny-vit

!pip install -r requirements.txt

Cloning into 'cs577-tiny-vit'...
remote: Enumerating objects: 428, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 428 (delta 128), reused 115 (delta 66), pack-reused 212 (from 1)
Receiving objects: 100% (428/428), 483.93 KiB | 741.00 KiB/s, done.
Resolving deltas: 100% (217/217), done.
/content/cs577-tiny-vit


## 3. Run tests
All tests should pass, they use fake data so no download.

In [4]:
#!pytest -q
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/cs577-tiny-vit
configfile: pytest.ini
testpaths: tests
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collected 49 items                                                             

tests/test_data.py::test_split_sizes_and_no_overlap PASSED               [  2%]
tests/test_data.py::test_split_is_stratified PASSED                      [  4%]
tests/test_data.py::test_split_is_deterministic PASSED                   [  6%]
tests/test_data.py::test_eval_transform_has_no_augmentation PASSED       [  8%]
tests/test_data.py::test_fake_loaders_shapes PASSED                      [ 10%]
tests/test_embeddings.py::test_patch_embedding_shape[4] PASSED           [ 12%]
tests/test_embeddings.py::test_patch_embedding_shape[8] PASSED           [ 14%]
tests/test_embeddings

## 4. Smoke run
Very small run on fake data, less than 1 minute.

In [5]:
!python main.py smoke
!ls -R results/

Epoch 1/1 done
results/:
ablation  profiling  sanity

results/ablation:

results/profiling:

results/sanity:
smoke_seed0.json


## 5. Sanity check
Tiny-batch overfit curve and shape trace for report.

In [6]:
!python main.py sanity
!python main.py trace --experiment baseline_p4

Shape trace for baseline_p4 (N = 1)
part           shape
input          (1, 3, 32, 32)
patch_embed    (1, 64, 128)
token_embed    (1, 65, 128)
patch_dropout  (1, 65, 128)
encoder        (1, 65, 128)
head           (1, 10)
saved results/sanity/shape_trace.csv


## 6. Patch-size ablation
Patch 4 vs 8, 3 seeds each. Takes about 1.5 hour. If Colab disconnects just run again, finished runs are skipped.

In [7]:
!python main.py ablate

100% 170M/170M [00:15<00:00, 11.0MB/s]
Epoch 1/30 done
Epoch 2/30 done
Epoch 3/30 done
Epoch 4/30 done
Epoch 5/30 done
Epoch 6/30 done
Epoch 7/30 done
Epoch 8/30 done
Epoch 9/30 done
Epoch 10/30 done
Epoch 11/30 done
Epoch 12/30 done
Epoch 13/30 done
Epoch 14/30 done
Epoch 15/30 done
Epoch 16/30 done
Epoch 17/30 done
Epoch 18/30 done
Epoch 19/30 done
Epoch 20/30 done
Epoch 21/30 done
Epoch 22/30 done
Epoch 23/30 done
Epoch 24/30 done
Epoch 25/30 done
Epoch 26/30 done
Epoch 27/30 done
Epoch 28/30 done
Epoch 29/30 done
Epoch 30/30 done
Epoch 1/30 done
Epoch 2/30 done
Epoch 3/30 done
Epoch 4/30 done
Epoch 5/30 done
Epoch 6/30 done
Epoch 7/30 done
Epoch 8/30 done
Epoch 9/30 done
Epoch 10/30 done
Epoch 11/30 done
Epoch 12/30 done
Epoch 13/30 done
Epoch 14/30 done
Epoch 15/30 done
Epoch 16/30 done
Epoch 17/30 done
Epoch 18/30 done
Epoch 19/30 done
Epoch 20/30 done
Epoch 21/30 done
Epoch 22/30 done
Epoch 23/30 done
Epoch 24/30 done
Epoch 25/30 done
Epoch 26/30 done
Epoch 27/30 done
Epoch 28/3

## 7. Efficiency profiling
Speed, peak memory, params, FLOPs for both patch sizes.

In [8]:
!python main.py profile

## 8. Show results
Tables from `results/` and figures from `docs/figures/`.

In [9]:
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 33 not upgraded.
Need to get 212 kB of archives.
After this operation, 735 kB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 poppler-utils amd64 24.02.0-1ubuntu9.10
Err:1 http://security.ubuntu.com/ubuntu noble-updates/main amd64 poppler-utils amd64 24.02.0-1ubuntu9.10
  404  Not Found [IP: 91.189.92.23 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/poppler/poppler-utils_24.02.0-1ubuntu9.10_amd64.deb  404  Not Found [IP: 91.189.92.23 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [10]:
# Display experiment result tables
display(pd.read_csv("results/ablation/summary.csv"))
display(pd.read_csv("results/profiling/profile.csv"))

# Display generated figures
for path in sorted(Path("docs/figures").glob("*.pdf")):
    print(path.name)

,experiment,patch_size,num_tokens,n_seeds,test_acc_mean,test_acc_std,val_acc_final_mean,train_time_mean_s,num_params
0,ablation_p8,8,17,3,0.696667,0.000551,0.707133,295.511665,1218186
1,baseline_p4,4,65,3,0.785833,0.002485,0.786333,295.372104,1205898


,experiment,patch_size,num_tokens,device,gpu_name,mode,batch_size,warmup_iters,runs,ms_per_batch_mean,ms_per_batch_std,images_per_s,peak_mem_mb,params,gflops_per_image
0,baseline_p4,4,65,cuda,NVIDIA L4,inference,128,10,5,6.261269,0.032074,20443.140429,63.230469,1205898,0.167122
1,baseline_p4,4,65,cuda,NVIDIA L4,train,128,10,5,19.988824,0.493251,6403.578304,506.782227,1205898,0.167122
2,ablation_p8,8,17,cuda,NVIDIA L4,inference,128,10,5,4.177102,0.011192,30643.250987,35.148438,1218186,0.041785
3,ablation_p8,8,17,cuda,NVIDIA L4,train,128,10,5,16.961708,0.381066,7546.410000,151.041992,1218186,0.041785


ablation_acc.pdf
overfit.pdf
training_curves.pdf


In [11]:
!ls -R results
!ls -R docs/figures

results:
ablation  profiling  sanity

results/ablation:
ablation_p8_seed0.json	baseline_p4_seed0.json	summary.csv
ablation_p8_seed1.json	baseline_p4_seed1.json
ablation_p8_seed2.json	baseline_p4_seed2.json

results/profiling:
profile.csv

results/sanity:
overfit_curve.json  shape_trace.csv  smoke_seed0.json
docs/figures:
ablation_acc.pdf  overfit.pdf  profile_table.tex  training_curves.pdf


In [12]:
!zip -r experiment_results.zip results/ docs/figures/

  adding: results/ (stored 0%)
  adding: results/sanity/ (stored 0%)
  adding: results/sanity/smoke_seed0.json (deflated 64%)
  adding: results/sanity/overfit_curve.json (deflated 66%)
  adding: results/sanity/.gitkeep (stored 0%)
  adding: results/sanity/shape_trace.csv (deflated 52%)
  adding: results/profiling/ (stored 0%)
  adding: results/profiling/.gitkeep (stored 0%)
  adding: results/profiling/profile.csv (deflated 47%)
  adding: results/ablation/ (stored 0%)
  adding: results/ablation/ablation_p8_seed1.json (deflated 76%)
  adding: results/ablation/ablation_p8_seed0.json (deflated 76%)
  adding: results/ablation/.gitkeep (stored 0%)
  adding: results/ablation/baseline_p4_seed1.json (deflated 76%)
  adding: results/ablation/summary.csv (deflated 40%)
  adding: results/ablation/baseline_p4_seed0.json (deflated 76%)
  adding: results/ablation/baseline_p4_seed2.json (deflated 76%)
  adding: results/ablation/ablation_p8_seed2.json (deflated 76%)
  adding: docs/figures/ (stored 0%)


In [13]:
from google.colab import files

files.download("experiment_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>